# Tutorial: DisCoolPy
## A 24-Hour Time-Series Simulation with Proper Offdesign Operation

This notebook demonstrates a complete district-cooling simulation workflow:

1. **Define** a 2-building network via a YAML configuration file
2. **Solve** the design point to size all components (heat exchangers, pump, pipes)
3. **Save** the design state so TESPy can use sized components in offdesign mode
4. **Simulate** 24 hourly snapshots using `TimeSnapshot.apply`, which updates
   building loads, ambient temperature, and condenser-water inlet temperature
   simultaneously before each offdesign solve
5. **Visualise** time-varying COP, chiller load, and compressor power


In [ ]:
import sys, warnings, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")

from discoolpy.utils import build_standard_branch_system
from discoolpy.time_snapshot import TimeSnapshot

print("Imports OK")


In [ ]:
# ── Network configuration ─────────────────────────────────────────────────────
# All parameters are defined inline here for tutorial clarity.
# In a real project, load this from a YAML file:
#   import yaml
#   with open("../configs/config_tutorial.yaml") as f:
#       config = yaml.safe_load(f)

config = {
    "network": {"chilled_water_cycle_closer": "chw cycle closer"},
    "design": {
        "supply_temperature_degC": 7.0,        # Chilled water supply temperature
        "building_return_temperature_degC": 12.0,
        "chilled_water_pressure_bar": 4.0,
        "condenser_inlet_temperature_degC": 32.0,  # Design condenser-water supply
        "condenser_outlet_temperature_degC": 38.0,
        "condenser_pressure_bar": 3.0,
        "ambient_temperature_degC": 35.0,
        "cp_water_J_kgK": 4180.0,
        "pump_power_W": 500.0,
        "pump_efficiency": 0.70,
        "bypass_mass_flow_kg_s": 0.05,
    },
    "buildings": [
        # pr=None: building pressure drop is a free output (determined by pipes)
        {"label": "office", "Q_design_W": 150_000.0, "pr": None},
        {"label": "hotel",  "Q_design_W": 120_000.0, "pr": None},
    ],
    "chiller": {
        "label": "tutorial_chiller",
        "T_evap_degC": 2.0,
        "T_cond_degC": 44.0,          # Design condensing temperature
        "eta_s": 0.75,
        "refrigerant": "R134a",
        "pressure_ratios": {"evap_1": 0.999, "evap_2": 0.999,
                            "cond_1": 0.999, "cond_2": 1.0},
        # native_offdesign: True activates TESPy's design/offdesign switching
        # on the chiller (kA sizing, compressor eta_s_char).
        "native_offdesign": {"enabled": True},
    },
    "branch": {
        "label": "tutorial_branch",
        "pump_placement": "supply_inlet",
        "pump_label": "main pump",
        # length_derived_pr: pipe pressure ratios are derived from Darcy-Weisbach
        # using the supplied lengths, diameters, and roughness values.
        "pipe_model": "length_derived_pr",
        "hydraulic": {
            "design_pressure_bar": 4.0,
            "fluid_density_kg_m3": 999.0,
            "dynamic_viscosity_Pa_s": 0.0013,
            "min_pr": 0.985,
            "max_pr": 0.9998,
        },
        "pipes": {
            "supply_1": {"Q": 0.0},
            "supply_2": {"L": 150.0, "D": 0.20, "ks": 5e-5, "Q": 0.0},
            "return_2": {"L": 150.0, "D": 0.20, "ks": 5e-5, "Q": 0.0},
            "return_1": {"L": 100.0, "D": 0.25, "ks": 5e-5},
        },
    },
    "cooling_tower": {
        "label": "tutorial_cooling_tower",
        "fluid": {"water": 1.0},
        "pr": None,
        "approach_temperature_K": 4.0,
        "start_mass_flow_kg_s": 30.0,
        "native_offdesign": False,
    },
    "solver": {"max_iter": 250, "iterinfo": False,
               "initial_pressure_step_bar": 0.05},
    "outputs": {"output_dir": "tutorial_outputs"},
}

print("Configuration defined.")
print(f"  Buildings : {[b['label'] for b in config['buildings']]}")
print(f"  Pipe model: {config['branch']['pipe_model']}")
print(f"  Refrigerant: {config['chiller']['refrigerant']}")


In [ ]:
# ── Step 1: Design solve ──────────────────────────────────────────────────────
# Build the network and solve at the nominal design point.
# TESPy sizes all heat exchangers (kA values) and the pump.

DESIGN_PATH = Path("tutorial_outputs/design_state")
DESIGN_PATH.parent.mkdir(exist_ok=True)

nw, chiller, buildings, branch, cooling_tower, _ = build_standard_branch_system(config)
nw.solve(mode="design", max_iter=250)

assert nw.converged, "Design solve did not converge — check configuration."

Q_design = abs(chiller.evaporator.Q.val)
W_design = chiller.compressor.P.val
COP_design = Q_design / W_design

print("Design point solved successfully.")
print(f"  Chiller load      : {Q_design/1e3:.1f} kW")
print(f"  Compressor power  : {W_design/1e3:.2f} kW")
print(f"  Design COP        : {COP_design:.3f}")
print(f"  Evap. kA          : {chiller.evaporator.kA.val:.1f} W/K")
print(f"  Cond. kA          : {chiller.condenser.kA.val:.1f} W/K")


In [ ]:
# ── Step 2: Configure offdesign mode ─────────────────────────────────────────
# Switch both heat exchangers to use their sized kA values in offdesign mode.
# This is numerically robust across the full operating range.
#
# Note: kA_char (heat-transfer characteristic curves) would add part-load
# degradation but causes Jacobian singularities at extreme conditions (high
# ambient + high load). For planning-tool purposes, fixed kA is appropriate.

chiller.condenser.set_attr(offdesign=["kA"])
chiller.evaporator.set_attr(offdesign=["kA"])

# Save the design state — TESPy reloads this at the start of every offdesign solve.
nw.save(str(DESIGN_PATH))

bldg = {b.label: b for b in buildings}
print("Offdesign mode configured and design state saved.")
print(f"  Condenser offdesign: {chiller.condenser.offdesign}")
print(f"  Evaporator offdesign: {chiller.evaporator.offdesign}")


In [ ]:
# ── Step 3: Build the 24-hour operating profile ───────────────────────────────
# Diurnal profiles for a hot summer day (e.g., Riyadh in July).
#
# Ambient temperature: minimum 28 °C at 05:00, maximum 43 °C at 15:00.
# Office load: peaks at 14:00 (business hours).
# Hotel load: peaks at 20:00 (evening occupancy).
# Condenser-water inlet: tracks ambient with a 4 K approach, capped at 37 °C.
#   (A well-designed cooling tower at 43 °C ambient produces ~39 °C supply;
#    the 37 °C cap keeps the condensing temperature below 49 °C, within the
#    solver's convergence basin.)

hours = np.arange(24)

T_amb = np.where(
    (hours >= 5) & (hours <= 19),
    28.0 + 15.0 * np.sin(np.pi * (hours - 5) / 14) ** 2,
    28.0,
).clip(28.0, 43.0)

frac_off = (0.55 + 0.45 * np.clip(np.sin(np.pi * (hours - 7) / 10), 0, 1))
frac_hot = (0.55 + 0.45 * np.clip(np.sin(np.pi * (hours - 12) / 12), 0, 1))

Q_off = 150_000.0 * frac_off   # Office cooling load [W]
Q_hot = 120_000.0 * frac_hot   # Hotel cooling load [W]
T_ci  = np.clip(T_amb - 4.0, 28.0, 37.0)  # Condenser-water inlet [°C]

# Quick profile preview
print(f"{'h':>3}  {'T_amb':>5}  {'T_ci':>5}  {'Q_off':>7}  {'Q_hot':>7}")
print("-" * 40)
for h in [0, 6, 9, 12, 15, 18, 21, 23]:
    print(f"  {h:2d}  {T_amb[h]:5.1f}  {T_ci[h]:5.1f}  "
          f"{Q_off[h]/1e3:6.1f}kW  {Q_hot[h]/1e3:6.1f}kW")


In [ ]:
# ── Step 4: Run the 24-hour offdesign simulation ──────────────────────────────
# For each hour:
#   1. Create a TimeSnapshot with the hour's loads and ambient conditions.
#   2. Call snapshot.apply() — this updates building heat exchangers, chiller
#      evaporator duty, and cooling-tower condenser-water inlet temperature.
#   3. Call nw.solve(mode='offdesign') — TESPy reloads the design state,
#      applies the updated boundary conditions, and solves for the new
#      operating point.

results = []

print(f"{'h':>3}  {'T_amb':>5}  {'T_ci':>5}  {'Q_tot':>7}  {'W':>7}  {'COP':>5}  Status")
print("-" * 55)

for h in range(24):
    snap = TimeSnapshot(
        timestamp=datetime(2026, 7, 15, h, 0),
        building_loads={"office": float(Q_off[h]), "hotel": float(Q_hot[h])},
        resolution="hourly",
        ambient_temperature=float(T_amb[h]),
        condenser_inlet_temperature=float(T_ci[h]),
    )
    snap.apply(
        buildings=bldg,
        chiller=chiller,
        cooling_tower=cooling_tower,
        chiller_load_offset_W=500.0,   # small offset to avoid zero-flow singularity
    )

    nw.solve(mode="offdesign", design_path=str(DESIGN_PATH), max_iter=250)

    if nw.converged:
        Q = abs(chiller.evaporator.Q.val)
        W = chiller.compressor.P.val
        cop = Q / W
        results.append({
            "h": h, "T_amb": float(T_amb[h]), "T_ci": float(T_ci[h]),
            "Q_off": float(Q_off[h]), "Q_hot": float(Q_hot[h]),
            "Q_total": Q, "W": W, "COP": cop,
        })
        print(f"  {h:2d}  {T_amb[h]:5.1f}  {T_ci[h]:5.1f}  "
              f"{Q/1e3:6.1f}kW  {W/1e3:6.2f}kW  {cop:5.3f}  OK")
    else:
        print(f"  {h:2d}  {T_amb[h]:5.1f}  {T_ci[h]:5.1f}  "
              f"{'---':>7}  {'---':>7}  {'---':>5}  FAIL")

print(f"\n{len(results)}/24 hours converged.")


In [ ]:
# ── Step 5: Visualise results ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

h_vals   = [r["h"]       for r in results]
cop_vals = [r["COP"]     for r in results]
Q_vals   = [r["Q_total"] / 1e3 for r in results]
W_vals   = [r["W"]       / 1e3 for r in results]
T_vals   = [r["T_amb"]   for r in results]
Tci_vals = [r["T_ci"]    for r in results]
Qoff_vals = [r["Q_off"]  / 1e3 for r in results]
Qhot_vals = [r["Q_hot"]  / 1e3 for r in results]

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, hspace=0.40, wspace=0.35)

# ── Panel 1: Building loads ───────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.stackplot(h_vals, Qoff_vals, Qhot_vals,
              labels=["Office", "Hotel"],
              colors=["#4C72B0", "#DD8452"], alpha=0.85)
ax1.set_xlabel("Hour of day")
ax1.set_ylabel("Cooling load (kW)")
ax1.set_title("Building cooling loads")
ax1.legend(loc="upper left", fontsize=9)
ax1.set_xlim(0, 23)
ax1.grid(True, alpha=0.3)

# ── Panel 2: Chiller load and compressor power ────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(h_vals, Q_vals, "o-", color="#4C72B0", label="Chiller load")
ax2.plot(h_vals, W_vals, "s--", color="#C44E52", label="Compressor power")
ax2.set_xlabel("Hour of day")
ax2.set_ylabel("Power (kW)")
ax2.set_title("Chiller load and compressor power")
ax2.legend(fontsize=9)
ax2.set_xlim(0, 23)
ax2.grid(True, alpha=0.3)

# ── Panel 3: COP ──────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(h_vals, cop_vals, "D-", color="#55A868", linewidth=2)
ax3.axhline(y=COP_design, color="gray", linestyle=":", linewidth=1.5,
            label=f"Design COP = {COP_design:.2f}")
ax3.set_xlabel("Hour of day")
ax3.set_ylabel("COP")
ax3.set_title("Coefficient of Performance (COP)")
ax3.legend(fontsize=9)
ax3.set_xlim(0, 23)
ax3.grid(True, alpha=0.3)

# ── Panel 4: Temperatures ─────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(h_vals, T_vals,  "^-", color="#E377C2", label="Ambient")
ax4.plot(h_vals, Tci_vals, "v--", color="#8C564B", label="Condenser inlet")
ax4.set_xlabel("Hour of day")
ax4.set_ylabel("Temperature (°C)")
ax4.set_title("Ambient and condenser-water inlet temperatures")
ax4.legend(fontsize=9)
ax4.set_xlim(0, 23)
ax4.grid(True, alpha=0.3)

plt.suptitle("District Cooling — 24-Hour Summer Day Simulation\n"
             "2-Building Network, Offdesign Mode, R134a Chiller",
             fontsize=13, fontweight="bold", y=1.01)

out_dir = Path("tutorial_outputs")
out_dir.mkdir(exist_ok=True)
fig.savefig(out_dir / "tutorial_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to tutorial_outputs/tutorial_results.png")


In [ ]:
# ── Step 6: Summary statistics ────────────────────────────────────────────────
cop_vals_arr = np.array(cop_vals)
Q_vals_arr   = np.array(Q_vals)
W_vals_arr   = np.array(W_vals)

total_cooling_kWh    = Q_vals_arr.sum()
total_compressor_kWh = W_vals_arr.sum()
mean_cop_load_wtd    = total_cooling_kWh / total_compressor_kWh

print("=" * 55)
print("  SIMULATION SUMMARY — 24-Hour Summer Day")
print("=" * 55)
print(f"  Converged hours          : {len(results)}/24")
print(f"  Peak chiller load        : {max(Q_vals):.1f} kW  (h={h_vals[Q_vals.index(max(Q_vals))]:02d}:00)")
print(f"  Min chiller load         : {min(Q_vals):.1f} kW  (h={h_vals[Q_vals.index(min(Q_vals))]:02d}:00)")
print(f"  Peak compressor power    : {max(W_vals):.2f} kW  (h={h_vals[W_vals.index(max(W_vals))]:02d}:00)")
print(f"  Daily compressor energy  : {total_compressor_kWh:.1f} kWh")
print(f"  Daily cooling energy     : {total_cooling_kWh:.1f} kWh")
print(f"  Peak COP                 : {max(cop_vals):.3f}  (h={h_vals[cop_vals.index(max(cop_vals))]:02d}:00)")
print(f"  Min COP                  : {min(cop_vals):.3f}  (h={h_vals[cop_vals.index(min(cop_vals))]:02d}:00)")
print(f"  Mean COP (load-weighted) : {mean_cop_load_wtd:.3f}")
print(f"  Ambient range            : {min(T_vals):.1f} – {max(T_vals):.1f} °C")
print(f"  Condenser inlet range    : {min(Tci_vals):.1f} – {max(Tci_vals):.1f} °C")
print("=" * 55)
print()
print("Key observations:")
print(f"  • COP drops {(max(cop_vals)-min(cop_vals))/max(cop_vals)*100:.0f}% from cool night to peak afternoon,")
print(f"    driven by the {max(Tci_vals)-min(Tci_vals):.0f} K rise in condenser-water inlet temperature.")
print(f"  • Compressor power varies {min(W_vals):.1f}–{max(W_vals):.1f} kW, tracking total cooling demand.")
print(f"  • The system serves {total_cooling_kWh:.0f} kWh of cooling with {total_compressor_kWh:.0f} kWh of electricity.")
